# 01 — Problema e benchmark

## Objetivo

Qual referência simples deve orientar as próximas etapas?

Comparamos Regressão Logística e Random Forest na mesma validação. O teste fica
reservado até o notebook final.

In [55]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import time
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.auxiliares import COLUNAS_NOMINAIS, carregar_base_preparada
from src.visual_utils import grafico_comparacao_modelos

dados = carregar_base_preparada(RAIZ)

### Visão rápida da base

In [56]:
dados = carregar_base_preparada(RAIZ)

display(dados.head())

resumo_base = pd.Series({
    "registros": len(dados),
    "variaveis": dados.shape[1],
    "valores_ausentes": int(dados.isna().sum().sum()),
    "duplicatas_exatas": int(dados.duplicated().sum()),
    "taxa_inadimplencia": f"{dados['inadimplente'].mean():.2%}",
})

display(resumo_base.to_frame("resultado"))

variaveis_exemplo = [
    "limite_credito",
    "idade",
    "valor_fatura_set",
    "valor_pago_set",
]

display(
    dados[variaveis_exemplo]
    .agg(["min", "median", "max"])
    .T
)

,id_cliente,limite_credito,sexo,escolaridade,estado_civil,idade,status_pagamento_set,status_pagamento_ago,status_pagamento_jul,status_pagamento_jun,...,valor_fatura_jun,valor_fatura_mai,valor_fatura_abr,valor_pago_set,valor_pago_ago,valor_pago_jul,valor_pago_jun,valor_pago_mai,valor_pago_abr,inadimplente
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


,resultado
registros,30000
variaveis,25
valores_ausentes,0
duplicatas_exatas,0
taxa_inadimplencia,22.12%


,min,median,max
limite_credito,10000.0,140000.0,1000000.0
idade,21.0,34.0,79.0
valor_fatura_set,-165580.0,22381.5,964511.0
valor_pago_set,0.0,2100.0,873552.0


## 1.1 — Como separar features, target e conjuntos?

In [57]:
X = dados.drop(columns=['id_cliente','inadimplente'] )
y = dados['inadimplente']

X_treino_validacao, X_teste, y_treino_validacao, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_treino_validacao,
    y_treino_validacao,
    test_size=0.25,
    random_state=42
)

Agora que a divisão está definida, salvamos os três conjuntos. Assim, os
próximos experimentos usam exatamente os mesmos dados sem repetir o split.

In [58]:
pasta_processados = RAIZ / "data" / "processed"
pasta_processados.mkdir(parents=True, exist_ok=True)

dados_treino = X_treino.copy()
dados_treino["inadimplente"] = y_treino

dados_validacao = X_validacao.copy()
dados_validacao["inadimplente"] = y_validacao

dados_teste = X_teste.copy()
dados_teste["inadimplente"] = y_teste

dados_treino.to_csv(pasta_processados / "treino.csv", index=False)
dados_validacao.to_csv(pasta_processados / "validacao.csv", index=False)
dados_teste.to_csv(pasta_processados / "teste.csv", index=False)

pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "linhas": [len(dados_treino), len(dados_validacao), len(dados_teste)],
    "proporcao": [len(dados_treino), len(dados_validacao), len(dados_teste)],
    "taxa_inadimplencia": [y_treino.mean(), y_validacao.mean(), y_teste.mean()],
}).assign(proporcao=lambda tabela: tabela["proporcao"] / tabela["linhas"].sum())

,conjunto,linhas,proporcao,taxa_inadimplencia
0,treino,18000,0.6,0.219778
1,validação,6000,0.2,0.225500
2,teste,6000,0.2,0.221167


O split estratificado produz 60% para treino, 20% para validação e 20% para
teste. ID e target não entram nas 23 features.

## 1.2 — O que a Regressão Logística entrega?

In [59]:
preprocessamento_logistico = ColumnTransformer(
    [(
        "nominais",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        COLUNAS_NOMINAIS,
    )],
    remainder=StandardScaler(),
    verbose_feature_names_out=False
)

modelo_logistico = Pipeline([
    ("preprocessamento", preprocessamento_logistico),
    ("modelo" , LogisticRegression(max_iter=2000, random_state=42))
])

inicio = time.perf_counter()
modelo_logistico.fit(X_treino, y_treino)
tempo_logistico = time.perf_counter() - inicio

previsoes_logisticas = modelo_logistico.predict(X_validacao)
probabilidades_logisticas = modelo_logistico.predict_proba(X_validacao)[:,1]

ap_logistica = average_precision_score(
    y_validacao,
    probabilidades_logisticas,
)

In [60]:
pd.DataFrame({
    "classe_prevista": previsoes_logisticas[:5],
    "probabilidade": probabilidades_logisticas[:5],
})

,classe_prevista,probabilidade
0,0,0.160336
1,1,0.559572
2,0,0.219828
3,0,0.268706
4,0,0.151278


In [61]:
print(f"Average Precision: {ap_logistica:.3f}")

Average Precision: 0.500


## 1.3 — O bagging melhora a referência?

In [62]:
preprocessamento_arvores = ColumnTransformer(
    [(
        "nominais",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        COLUNAS_NOMINAIS,
    )],
    remainder="passthrough",
    verbose_feature_names_out=False
)

modelo_random_forest = Pipeline([
    ("preprocessamento", preprocessamento_arvores),
    ("modelo" , RandomForestClassifier(n_estimators=200, 
                                       random_state=42,
                                       n_jobs=-1))
])

inicio = time.perf_counter()
modelo_random_forest.fit(X_treino, y_treino)
tempo_random_forest = time.perf_counter() - inicio

previsoes_random_forest = modelo_random_forest.predict(X_validacao)
probabilidades_random_forest = modelo_random_forest.predict_proba(X_validacao)[:,1]

ap_random_forest = average_precision_score(
    y_validacao,
    probabilidades_random_forest,
)



## 1.4 — Como comparar os dois modelos?

In [64]:
resultados = pd.DataFrame([
    {
    "modelo": "Regressão Logística",
    "average_precision": ap_logistica,
    "tempo_treino_s":tempo_logistico,
    },
    {
    "modelo": "Random Forest",
    "average_precision": ap_random_forest,
    "tempo_treino_s":tempo_random_forest,        
    }
            ])

In [65]:
fig = grafico_comparacao_modelos(resultados)
fig.show()

In [66]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparacao_metricas = pd.DataFrame([
    {
        "modelo": "Regressão Logística",
        "AP": ap_logistica,
        "Precision": precision_score(y_validacao, previsoes_logisticas),
        "Recall": recall_score(y_validacao, previsoes_logisticas),
        "F1": f1_score(y_validacao, previsoes_logisticas),
    },
    {
        "modelo": "Random Forest",
        "AP": ap_random_forest,
        "Precision": precision_score(y_validacao, previsoes_random_forest),
        "Recall": recall_score(y_validacao, previsoes_random_forest),
        "F1": f1_score(y_validacao, previsoes_random_forest),
    },
])

comparacao_metricas.round(3)

,modelo,AP,Precision,Recall,F1
0,Regressão Logística,0.500,0.718,0.224,0.341
1,Random Forest,0.526,0.658,0.358,0.463


## 1.5 — Resultado

A Random Forest melhora o ranking probabilístico em relação à logística e será
o exemplo prático de bagging. Precision, Recall e F1 ainda usam limiar 0,50.